In [ ]:
%load_ext autoreload
%autoreload 2

import time
from typing import Any
import numpy as np
import pylab as plt 
import networkx as nx 
import pandas as pd 
from sklearn.metrics import pairwise_distances




In [ ]:
# Torus module imports
from modules import geometry
from modules import graphio
from modules import metrics
from modules import visualization as vis
from modules.projector import MDSTorusProjector
from standalone_toruslayout.toruslayout import TorusLayoutResult
from standalone_toruslayout.toruslayout import layout_graph

In [ ]:
graphs = [
    ("grid_5x5", *graphio.get_periodic_lattice(5, 5)),
    ("grid_10x10", *graphio.get_periodic_lattice(10, 10)),
    # ("grid_20x20", *graphio.get_periodic_lattice(20, 20)),
]


In [ ]:
rows = []
fig, axes = plt.subplots(len(graphs), 3, figsize=(12, 4 * len(graphs)))
axes = np.atleast_2d(axes)

for row_idx, (name, G, D) in enumerate(graphs):
    nx_size = max(i for i, j in G.nodes()) + 1
    ny_size = max(j for i, j in G.nodes()) + 1
    X_given = np.array([[i / nx_size, j / ny_size] for i, j in G.nodes()], dtype=float)
    alpha_given = metrics.estimate_alpha(X_given, D)
    scaled_given = lambda p, q, a=alpha_given: a * geometry.torus_distance(p, q)

    t0 = time.perf_counter()
    wrap = layout_graph(
        G,
        shortest_path_lengths=D,
    )
    wrap_runtime = time.perf_counter() - t0
    X_wrap = wrap.positions
    alpha_wrap = metrics.estimate_alpha(X_wrap, D)
    scaled_wrap = lambda p, q, a=alpha_wrap: a * geometry.torus_distance(p, q)

    proj = MDSTorusProjector()
    t0 = time.perf_counter()
    X_mds = proj.fit_transform(D)
    mds_runtime = time.perf_counter() - t0
    alpha_mds = proj.alpha_
    scaled_mds = lambda p, q, a=alpha_mds: a * geometry.torus_distance(p, q)

    rows.append({
        "graph": name,
        "alpha_given": round(alpha_given, 3),
        "stress_given": round(metrics.geodesic_stress(X_given, D, scaled_given), 4),
        "sgs_given": round(metrics.SGS(X_given, D, scaled_given), 4),
        "alpha_wrap": round(alpha_wrap, 3),
        "stress_wrap": round(metrics.geodesic_stress(X_wrap, D, scaled_wrap), 4),
        "sgs_wrap": round(metrics.SGS(X_wrap, D, scaled_wrap), 4),
        "iters_wrap": wrap.iterations,
        "runtime_wrap_s": round(wrap_runtime, 3),
        "alpha_mds": round(alpha_mds, 3),
        "stress_mds": round(metrics.geodesic_stress(X_mds, D, scaled_mds), 4),
        "sgs_mds": round(metrics.SGS(X_mds, D, scaled_mds), 4),
        "runtime_mds_s": round(mds_runtime, 3),
    })

    vis.plot_embedding_with_torus_edges(X_given, G, ax=axes[row_idx, 0], edge_alpha=0.7)
    vis.plot_embedding_with_torus_edges(X_wrap, G, ax=axes[row_idx, 1], edge_alpha=0.7)
    vis.plot_embedding_with_torus_edges(X_mds, G, ax=axes[row_idx, 2], edge_alpha=0.7)
    axes[row_idx, 0].invert_yaxis()
    axes[row_idx, 1].invert_yaxis()
    axes[row_idx, 2].invert_yaxis()
    axes[row_idx, 0].set_title(f"Graph {name} - given drawing")
    axes[row_idx, 1].set_title(f"Graph {name} - standalone torus layout")
    axes[row_idx, 2].set_title(f"Graph {name} - MDS torus")

plt.tight_layout()
plt.show()

pd.DataFrame(rows)


In [ ]:
chen_graphs = graphio.load_chen_graphs("chengraphs/*.json")
chen_graphs = sorted(chen_graphs, key = lambda x: int(x[0]))

In [ ]:
chen_graphs_set = chen_graphs[:2]
rows = []
fig, axes = plt.subplots(len(chen_graphs_set), 2, figsize=(8, 4 * len(chen_graphs_set)))
axes = np.atleast_2d(axes)

for row_idx, (name, G, X_given, D) in enumerate(chen_graphs_set):
    t0 = time.perf_counter()
    wrap = layout_graph(G, shortest_path_lengths=D)
    wrap_runtime = time.perf_counter() - t0
    X_wrap = wrap.positions
    alpha_wrap = metrics.estimate_alpha(X_wrap, D)
    scaled_wrap = lambda p, q, a=alpha_wrap: a * geometry.torus_distance(p, q)

    proj = MDSTorusProjector()
    t0 = time.perf_counter()
    X_mds = proj.fit_transform(D)
    mds_runtime = time.perf_counter() - t0
    alpha_mds = proj.alpha_
    scaled_mds = lambda p, q, a=alpha_mds: a * geometry.torus_distance(p, q)

    row = {
        "graph": name,
        "alpha_wrap": round(alpha_wrap, 3),
        "stress_wrap": round(metrics.geodesic_stress(X_wrap, D, scaled_wrap), 4),
        "sgs_wrap": round(metrics.SGS(X_wrap, D, scaled_wrap), 4),
        "iters_wrap": wrap.iterations,
        "runtime_wrap_s": round(wrap_runtime, 3),
        "alpha_mds": round(alpha_mds, 3),
        "stress_mds": round(metrics.geodesic_stress(X_mds, D, scaled_mds), 4),
        "sgs_mds": round(metrics.SGS(X_mds, D, scaled_mds), 4),
        "runtime_mds_s": round(mds_runtime, 3),
    }
    if X_given is not None:
        alpha_given = metrics.estimate_alpha(X_given, D)
        scaled_given = lambda p, q, a=alpha_given: a * geometry.torus_distance(p, q)
        row.update({
            "alpha_given": round(alpha_given, 3),
            "stress_given": round(metrics.geodesic_stress(X_given, D, scaled_given), 4),
            "sgs_given": round(metrics.SGS(X_given, D, scaled_given), 4),
        })

    rows.append(row)

    vis.plot_embedding_with_torus_edges(X_wrap, G, ax=axes[row_idx, 0], edge_alpha=0.5)
    vis.plot_embedding_with_torus_edges(X_mds, G, ax=axes[row_idx, 1], edge_alpha=0.5)
    axes[row_idx, 0].invert_yaxis()
    axes[row_idx, 1].invert_yaxis()
    axes[row_idx, 0].set_title(f"Graph {name} - standalone torus layout")
    axes[row_idx, 1].set_title(f"Graph {name} - MDS torus")

plt.tight_layout()
plt.show()
df = pd.DataFrame(rows)


In [ ]:
from IPython.display import display

def summarize_side_by_side(df, *, left_suffix="wrap", right_suffix="mds"):
    analysis_df = df.copy()
    summary_rows = []
    metrics = ["alpha", "stress", "distortion", "goodness", "sgs", "NP", "runtime"]

    for metric in metrics:
        if metric != "runtime":
            left_col = f"{metric}_{left_suffix}"
            right_col = f"{metric}_{right_suffix}"
        else:
            left_col = f"{metric}_{left_suffix}_s"
            right_col = f"{metric}_{right_suffix}_s"
        if left_col not in analysis_df.columns or right_col not in analysis_df.columns:
            continue

        delta_col = f"{metric}_delta"
        analysis_df[delta_col] = analysis_df[left_col] - analysis_df[right_col]
        # if metric != "runtime":
        analysis_df[f"{metric}_delta_pct"] = 100 * analysis_df[delta_col] / analysis_df[right_col]

        right_mean = analysis_df[right_col].mean()
        summary_rows.append({
            "metric": metric,
            f"mean_{left_suffix}": analysis_df[left_col].mean(),
            f"mean_{right_suffix}": right_mean,
            "mean_delta": analysis_df[delta_col].mean(),
            "mean_delta_pct": np.nan if right_mean == 0 else 100 * analysis_df[delta_col].mean() / right_mean,
            f"{left_suffix}_better_rows": int((analysis_df[delta_col] < 0).sum()),
        })

    return analysis_df, pd.DataFrame(summary_rows)

def display_side_by_side(df, *, left_suffix="wrap", right_suffix="mds"):
    analysis_df, summary_df = summarize_side_by_side(df, left_suffix=left_suffix, right_suffix=right_suffix)
    display(analysis_df)
    display(summary_df)
    return analysis_df, summary_df

def runtime_table(labels, left_runtime_s, right_runtime_s, *, label_name="cell_id", left_label="wrap", right_label="mdstorus"):
    return pd.DataFrame({
        label_name: list(labels),
        f"{left_label}_runtime_s": [round(left_runtime_s, 3)] * len(labels),
        f"{right_label}_runtime_s": [round(right_runtime_s, 3)] * len(labels),
    })

def plot_cell_profiles(left_X, right_X, spikes, cell_ids, *, left_label="wrap", right_label="mdstorus"):
    fig, axes = plt.subplots(2, len(cell_ids), figsize=(12, 5))
    for col, idx in enumerate(cell_ids):
        order = np.argsort(spikes[:, idx])
        axes[0, col].scatter(left_X[order, 0], left_X[order, 1], c=spikes[order, idx], s=2)
        axes[1, col].scatter(right_X[order, 0], right_X[order, 1], c=spikes[order, idx], s=2)
        axes[0, col].axis("off")
        axes[1, col].axis("off")
        axes[0, col].set_aspect("equal")
        axes[1, col].set_aspect("equal")
        axes[0, col].set_title(f"Cell {idx}")
        axes[1, col].set_title(f"Cell {idx}")
    axes[0, 0].set_ylabel(left_label)
    axes[1, 0].set_ylabel(right_label)
    return fig, axes

def plot_torus_pair(left_X, right_X, G, axes, *, left_title, right_title, colors=None, edge_alpha=0.5):
    vis.plot_embedding_with_torus_edges(left_X, G, ax=axes[0], colors=colors, edge_alpha=edge_alpha)
    vis.plot_embedding_with_torus_edges(right_X, G, ax=axes[1], colors=colors, edge_alpha=edge_alpha)
    axes[0].invert_yaxis()
    axes[1].invert_yaxis()
    axes[0].set_title(left_title)
    axes[1].set_title(right_title)
    return axes

analysis_df, summary_df = display_side_by_side(df, left_suffix="wrap", right_suffix="mds")


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class WrapMdstorusComparison:
    wrap: TorusLayoutResult
    projector: MDSTorusProjector
    X_wrap: np.ndarray
    X_mdstorus: np.ndarray
    wrap_runtime_s: float
    mdstorus_runtime_s: float
    wrap_metrics: dict[str, float]
    mdstorus_metrics: dict[str, float]

    def to_row(self, **metadata: Any) -> dict[str, Any]:
        row = dict(metadata)
        for metric in ("alpha", "stress", "distortion", "goodness", "NP"):
            row[f"{metric}_wrap"] = self.wrap_metrics[metric]
            row[f"{metric}_mdstorus"] = self.mdstorus_metrics[metric]
        row["runtime_wrap_s"] = round(self.wrap_runtime_s, 3)
        row["runtime_mdstorus_s"] = round(self.mdstorus_runtime_s, 3)
        row["iterations_wrap"] = int(self.wrap.iterations)
        row["iterations_mdstorus"] = np.nan
        return row

def compare_wrap_vs_mdstorus(G, D, *, wrap_graph=None):
    wrap_graph = G if wrap_graph is None else wrap_graph

    t0 = time.perf_counter()
    wrap = layout_graph(wrap_graph, shortest_path_lengths=D, node_memory_mb=8096)
    wrap_runtime_s = time.perf_counter() - t0
    X_wrap = wrap.positions
    alpha_wrap = metrics.estimate_alpha(X_wrap, D)
    scaled_wrap = lambda p, q, a=alpha_wrap: a * geometry.torus_distance(p, q)

    projector = MDSTorusProjector()
    t0 = time.perf_counter()
    X_mdstorus = projector.fit_transform(D)
    mdstorus_runtime_s = time.perf_counter() - t0
    alpha_mdstorus = projector.alpha_
    scaled_mdstorus = lambda p, q, a=alpha_mdstorus: a * geometry.torus_distance(p, q)

    wrap_metrics = {
        "alpha": alpha_wrap,
        "stress": metrics.geodesic_stress(X_wrap, D, scaled_wrap),
        "distortion": metrics.geodesic_distortion(X_wrap, D, scaled_wrap),
        "goodness": metrics.SGS(X_wrap, D, scaled_wrap),
        "NP": metrics.geodesic_NP(X_wrap, D, scaled_wrap),
        "runtime_wrap_s": wrap_runtime_s,
    }
    mdstorus_metrics = {
        "alpha": alpha_mdstorus,
        "stress": metrics.geodesic_stress(X_mdstorus, D, scaled_mdstorus),
        "distortion": metrics.geodesic_distortion(X_mdstorus, D, scaled_mdstorus),
        "goodness": metrics.SGS(X_mdstorus, D, scaled_mdstorus),
        "NP": metrics.geodesic_NP(X_mdstorus, D, scaled_mdstorus),
        "runtime_mdstorus_s": mdstorus_runtime_s,
    }

    return WrapMdstorusComparison(
        wrap=wrap,
        projector=projector,
        X_wrap=X_wrap,
        X_mdstorus=X_mdstorus,
        wrap_runtime_s=wrap_runtime_s,
        mdstorus_runtime_s=mdstorus_runtime_s,
        wrap_metrics=wrap_metrics,
        mdstorus_metrics=mdstorus_metrics,
    )


## grid_cells_4800

Wrap and MDSTorus comparison on the subsampled place-cell data, following the structure used in `examples.ipynb`.


In [ ]:
a = np.load("grid_cells_4800.npz")
data, spikes = metrics.subsample(5000, a["data"], a["spikes"])

D = pairwise_distances(data)
G = nx.empty_graph(D.shape[0])

comparison = compare_wrap_vs_mdstorus(G, D)
X_wrap = comparison.X_wrap
X_mdstorus = comparison.X_mdstorus

cell_ids = [5, 39, 75, 109, 115]
fig, axes = plot_cell_profiles(X_wrap, X_mdstorus, spikes, cell_ids, left_label="wrap", right_label="MDSTorus")
fig.suptitle(
    f"grid_cells_4800: wrap vs MDSTorus  (wrap {comparison.wrap_runtime_s:.3f}s, MDSTorus {comparison.mdstorus_runtime_s:.3f}s)",
    y=1.02,
)
plt.tight_layout()
plt.show()

grid_row = comparison.to_row(dataset="grid_cells_4800")
grid_df = pd.DataFrame([grid_row])
grid_analysis_df, grid_summary_df = summarize_side_by_side(grid_df, left_suffix="wrap", right_suffix="mdstorus")
grid_runtime_df = runtime_table(cell_ids, comparison.wrap_runtime_s, comparison.mdstorus_runtime_s)

display(grid_analysis_df)
display(grid_summary_df)
display(grid_runtime_df)


## Planted Partition

Wrap and MDSTorus comparison for planted partition block models.


In [ ]:
block_specs = [(4, 80, 0.2, 0.01), (5, 100, 0.2, 0.01), (7, 100, 0.2, 0.01)]
# block_specs = [(4, 80, 0.2, 0.01)]
rows = []
fig, axes = plt.subplots(len(block_specs), 2, figsize=(8, 4 * len(block_specs)))
axes = np.atleast_2d(axes)

for row_idx, (k, n_parts, p_in, p_out) in enumerate(block_specs):
    G = nx.planted_partition_graph(k, n_parts, p_in, p_out)
    D, nodes = graphio.apsp_distance_matrix(G)

    comparison = compare_wrap_vs_mdstorus(G, D)
    X_wrap = comparison.X_wrap
    X_mdstorus = comparison.X_mdstorus

    colors = np.repeat(np.arange(k), n_parts)
    plot_torus_pair(
        X_wrap,
        X_mdstorus,
        G,
        axes[row_idx],
        left_title=f"Planted partition k={k} - wrap ({comparison.wrap_runtime_s:.3f}s)",
        right_title=f"Planted partition k={k} - MDSTorus ({comparison.mdstorus_runtime_s:.3f}s)",
        colors=list(colors),
        edge_alpha=0.5,
    )

    rows.append(comparison.to_row(graph=f"k={k}", k=k, nodes=G.number_of_nodes(), edges=G.number_of_edges()))

plt.tight_layout()
plt.show()
block_df = pd.DataFrame(rows)
block_analysis_df, block_summary_df = summarize_side_by_side(block_df, left_suffix="wrap", right_suffix="mdstorus")
display(block_analysis_df)
display(block_summary_df)
